# FathomNet 2026 - Kaggle yolo26l 1024データセット版

## 0. セットアップ

In [ ]:
!pip install ensemble-boxes ultralytics -q

import os
import json
import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
from PIL import Image as PILImage
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
from pathlib import Path
from tqdm import tqdm
import yaml
import re
import shutil

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')

# データパス
DATA_DIR   = '/kaggle/input/datasets/kenkagkag/sinkai-k'
TRAIN_JSON = f'{DATA_DIR}/train_dataset.json'
TEST_JSON  = f'{DATA_DIR}/test_dataset.json'
TRAIN_IMG  = f'{DATA_DIR}/images/train'
TEST_IMG   = f'{DATA_DIR}/images/test'
TRAIN_LBL  = f'{DATA_DIR}/labels/train'

with open(TRAIN_JSON) as f:
    train_data = json.load(f)
with open(TEST_JSON) as f:
    test_data = json.load(f)

# カテゴリマッピング（trainから固定）
cat_ids = sorted([c['id'] for c in train_data['categories']])
category_to_yolo = {cat_id: i for i, cat_id in enumerate(cat_ids)}
cat_names = {c['id']: c['name'] for c in train_data['categories']}
yolo_to_category = cat_ids

# train/testカテゴリ一致確認
test_cat_ids = sorted([c['id'] for c in test_data['categories']])
assert cat_ids == test_cat_ids, 'カテゴリが一致しません！'
print('カテゴリ一致確認OK')

fname_to_id    = {os.path.basename(img['file_name']): img['id'] for img in test_data['images']}
img_id_to_info = {img['id']: img for img in test_data['images']}

test_files = sorted(os.listdir(TEST_IMG))
print(f'テスト画像数: {len(test_files)}')
print(f'カテゴリ数: {len(cat_ids)}')

## 1. データ準備（GroupKFold日付ベース分割）

In [ ]:
from sklearn.model_selection import GroupKFold

def make_group_id(img):
    raw_date = img.get('date_captured', None)
    if raw_date is not None:
        try:
            dt = pd.to_datetime(str(raw_date), errors='coerce')
            date_part = dt.strftime('%Y-%m-%d') if pd.notnull(dt) else 'nodate'
        except:
            date_part = 'nodate'
    else:
        date_part = 'nodate'
    stem = Path(img.get('file_name', '')).stem.lower()
    norm = re.sub(r'[^a-z0-9]+', '_', stem)
    prefix = re.sub(r'(_?\d+)$', '', norm).strip('_')
    if len(prefix) < 3:
        prefix = norm if norm else 'nofile'
    return f'{date_part}__{prefix}'

groups = [make_group_id(img) for img in train_data['images']]
kf = GroupKFold(n_splits=5)
splits = list(kf.split(np.arange(len(train_data['images'])), groups=groups))

fold = 0
idx_tr, idx_va = splits[fold]
print(f'train: {len(idx_tr)}枚, val: {len(idx_va)}枚')

In [ ]:
WORK = '/kaggle/working/dataset'
for d in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

def setup_fold(indices, split_name):
    for idx in tqdm(indices, desc=split_name):
        img = train_data['images'][idx]
        stem = Path(img['file_name']).stem

        # 画像をシンボリックリンク
        src_img = f'{TRAIN_IMG}/{stem}.jpg'
        dst_img = f'{WORK}/images/{split_name}/{stem}.jpg'
        if not os.path.exists(dst_img) and os.path.exists(src_img):
            try:
                os.symlink(src_img, dst_img)
            except:
                shutil.copy(src_img, dst_img)

        # ラベルをシンボリックリンク
        src_lbl = f'{TRAIN_LBL}/{stem}.txt'
        dst_lbl = f'{WORK}/labels/{split_name}/{stem}.txt'
        if not os.path.exists(dst_lbl) and os.path.exists(src_lbl):
            try:
                os.symlink(src_lbl, dst_lbl)
            except:
                shutil.copy(src_lbl, dst_lbl)

setup_fold(idx_tr, 'train')
setup_fold(idx_va, 'val')

# yaml作成
dataset_config = {
    'path': WORK,
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: cat_names[cat_id] for i, cat_id in enumerate(cat_ids)}
}
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(dataset_config, f, allow_unicode=True)

print(f'train画像: {len(os.listdir(f"{WORK}/images/train"))}枚')
print(f'val画像:   {len(os.listdir(f"{WORK}/images/val"))}枚')
print(f'trainラベル: {len(os.listdir(f"{WORK}/labels/train"))}件')
print(f'valラベル:   {len(os.listdir(f"{WORK}/labels/val"))}件')
with open('/kaggle/working/data.yaml') as f:
    print(f.read())

## 2. yolo26l 学習（imgsz=1024）

In [ ]:
YOLOL_PATH = '/kaggle/working/runs/yolol/weights/best.pt'

if os.path.exists(YOLOL_PATH):
    print('学習済みyolo26lが見つかりました。スキップします。')
else:
    print('yolo26lの学習を開始します...')
    model_train = YOLO('yolo26l.pt')
    model_train.train(
        data='/kaggle/working/data.yaml',
        epochs=30,
        imgsz=1024,
        batch=8,
        hsv_v=0.7,
        val=True,
        project='/kaggle/working/runs',
        name='yolol',
        exist_ok=True,
    )
    print('yolo26l学習完了！')
    del model_train
    torch.cuda.empty_cache()

## 3. CVスコア確認

In [ ]:
results_csv = pd.read_csv('/kaggle/working/runs/yolol/results.csv')
print(results_csv[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(10))
best_epoch = results_csv['metrics/mAP50-95(B)'].idxmax()
print(f'ベストepoch: {best_epoch+1}')
print(results_csv.iloc[best_epoch][['metrics/mAP50(B)', 'metrics/mAP50-95(B)']])

## 4. モデル読み込み

In [ ]:
model_yolol = YOLO(YOLOL_PATH)
print('yolo26l loaded')

## 5. 推論設定

In [ ]:
CONF             = 0.01
IOU              = 0.6
MAX_DET          = 1000
WBF_IOU          = 0.55
WBF_SKIP_BOX_THR = 0.005

TTA_PATTERNS = [
    (False, 1024),  # オリジナル
    (True,  1024),  # 水平反転
    (False,  896),  # スケール違い
]

## 6. TTA推論 & WBF

In [ ]:
def predict_with_tta(model, img_path, conf, iou):
    img_orig = PILImage.open(img_path).convert('RGB')
    all_boxes, all_scores, all_labels = [], [], []

    for do_flip, sz in TTA_PATTERNS:
        img = img_orig.copy()
        if do_flip:
            img = TF.hflip(img)

        result = model.predict(
            source=img,
            conf=conf,
            iou=iou,
            max_det=MAX_DET,
            imgsz=sz,
            verbose=False,
            half=True,
        )[0]

        if len(result.boxes) == 0:
            continue

        boxes  = result.boxes.xyxyn.cpu().numpy().tolist()
        scores = result.boxes.conf.cpu().numpy().tolist()
        labels = result.boxes.cls.cpu().numpy().astype(int).tolist()

        if do_flip:
            boxes = [[1-x2, y1, 1-x1, y2] for x1, y1, x2, y2 in boxes]

        boxes = [[min(max(v, 0.0), 1.0) for v in box] for box in boxes]
        all_boxes.append(boxes)
        all_scores.append(scores)
        all_labels.append(labels)

    if not all_boxes:
        return [], [], []

    boxes_f, scores_f, labels_f = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels,
        weights=[1.0] * len(all_boxes),
        iou_thr=WBF_IOU,
        skip_box_thr=WBF_SKIP_BOX_THR,
    )
    return boxes_f.tolist(), scores_f.tolist(), labels_f.tolist()


rows = []

for fname in tqdm(test_files):
    img_path = os.path.join(TEST_IMG, fname)
    image_id = fname_to_id.get(fname)
    if image_id is None:
        stem = Path(fname).stem
        for key in fname_to_id:
            if Path(key).stem == stem:
                image_id = fname_to_id[key]
                break
    if image_id is None:
        continue

    w = img_id_to_info[image_id]['width']
    h = img_id_to_info[image_id]['height']

    boxes_f, scores_f, labels_f = predict_with_tta(model_yolol, img_path, CONF, IOU)
    if len(boxes_f) == 0:
        continue

    for box, score, label in zip(boxes_f, scores_f, labels_f):
        x1, y1, x2, y2 = box
        rows.append({
            'image_id':    image_id,
            'category_id': yolo_to_category[int(label)],
            'bbox_x':      x1 * w,
            'bbox_y':      y1 * h,
            'bbox_width':  (x2 - x1) * w,
            'bbox_height': (y2 - y1) * h,
            'score':       float(score),
        })

print(f'予測数: {len(rows)}')
print(f'1画像あたり平均: {len(rows)/len(test_files):.1f}box')

## 7. 提出ファイル作成

In [ ]:
submission = pd.DataFrame(rows)
submission['annotation_id'] = np.arange(len(submission))
submission = submission[[
    'annotation_id', 'image_id', 'category_id',
    'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'score'
]]
submission['score'] = submission['score'].clip(0, 1)

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'完了！{len(submission)}行')
print(submission.head())